# Part 3: Bring in work context with Work IQ

Part 3 at Caldova begins with three emails about Summit Dose and product **CALD-201**: an equipment-mapping approval, an incomplete subcontractor package, and a decision about protecting the April delivery. The approved purchase order says one thing, and the inbox says what people are doing about it now. You need Work IQ to read that work context as you, then act on it by scheduling the decision and sending a follow-up.

**🎯 Mission**
- Use Work IQ's **Copilot tools** to get a synthesized answer with `ask` and the cited evidence behind it with `retrieve`
- Use the **discovery and read tools** to find paths, inspect schemas, and fetch structured email data
- Use the **write tools** to schedule a decision meeting and send a follow-up with the approved purchase order attached, then download it back

**Before you run:** Complete the browser steps for Exercise 3 in the lab instructions to read the three CALD-201 emails in Outlook. Then follow this notebook from top to bottom, running the code cells in sequence.

## The building blocks

Work IQ gives agents the signed-in user's Microsoft 365 work context, honoring their permissions and sensitivity labels. It speaks MCP at `https://workiq.svc.cloud.microsoft/mcp` and exposes four groups of tools:

| Group | Tools | Purpose |
| --- | --- | --- |
| Copilot | `ask`, `retrieve`, `list_agents` | A synthesized answer, cited grounding passages, and the agents you can ask |
| Discovery | `search_paths`, `get_schema` | Find resource paths and request schemas at runtime |
| Read | `fetch`, `fetch_blob`, `call_function` | Read entities, download files and attachments, and call read-only functions |
| Write | `create_entity`, `update_entity`, `delete_entity`, `do_action` | Create, change, and remove items, and run actions such as sending mail |

Every call is **delegated**. It runs as you, not as an application.

## Step 1: Sign in and create a Work IQ helper

Load the Work IQ app registration created during setup and acquire a delegated token with MSAL. Sign in with your lab account in the browser when prompted.

The `call_workiq` helper opens an MCP session, calls one tool, and returns its payload. Work IQ returns most results in the MCP `structuredContent` field, and a few tools return JSON text instead, so the helper handles both.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import json
import os

import httpx
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from msal import PublicClientApplication

load_dotenv(override=True)

WORK_IQ_MCP_URL = "https://workiq.svc.cloud.microsoft/mcp"
WORK_IQ_SCOPE = "api://workiq.svc.cloud.microsoft/WorkIQAgent.Ask"
app_id = os.environ["WORK_IQ_ENTRA_APP_ID"]
tenant_id = os.environ["WORK_IQ_ENTRA_TENANT_ID"]

app = PublicClientApplication(client_id=app_id, authority=f"https://login.microsoftonline.com/{tenant_id}")
auth_result = app.acquire_token_interactive([WORK_IQ_SCOPE])
if "access_token" not in auth_result:
    raise RuntimeError(f"Authentication failed: {auth_result.get('error_description', auth_result)}")

token = auth_result["access_token"]
lab_email = auth_result.get("id_token_claims", {}).get("preferred_username", "").strip()


async def call_workiq(tool_name, arguments=None):
    """Call one Work IQ tool and return its structured payload."""
    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {token}"},
        timeout=httpx.Timeout(30, read=180),
    ) as http_client:
        async with streamable_http_client(WORK_IQ_MCP_URL, http_client=http_client) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool(tool_name, arguments or {})
    if result.isError:
        raise RuntimeError(f"Work IQ {tool_name} failed: {result.content}")
    if result.structuredContent is not None:
        return result.structuredContent
    text = "".join(block.text for block in result.content if block.type == "text")
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return text


print(f"Signed in to Work IQ as {lab_email}")

## Step 2: List the available tools

Initialize an MCP session and list every tool with its required arguments. Keep this list in view. The rest of the notebook walks through each group.

In [ ]:
async with httpx.AsyncClient(
    headers={"Authorization": f"Bearer {token}"},
    timeout=httpx.Timeout(30, read=180),
) as http_client:
    async with streamable_http_client(WORK_IQ_MCP_URL, http_client=http_client) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_response = await session.list_tools()

workiq_tools = {tool.name: tool for tool in tools_response.tools}
for tool in tools_response.tools:
    first_sentence = (tool.description or "").split(". ")[0]
    print(f"{tool.name}: {first_sentence}")
    print("  Required arguments:", tool.inputSchema.get("required", []))

---

## A: Ask, retrieve, and read

## Step 3: Ask for a synthesized answer

`ask` sends a natural-language question to Microsoft 365 Copilot. Work IQ plans the retrieval and returns a synthesized answer with citations, plus a `conversationId` that you can pass back to ask a follow-up in the same conversation.

⏱️ Allow 30 to 60 seconds for each answer.

In [ ]:
from IPython.display import Markdown, display

question = (
    "Summarize my recent emails about CALD-201. For each email, who sent it, "
    "what action is open, who owns it, and what is the deadline?"
)
ask_result = await call_workiq("ask", {"question": question})
ask_answer = ask_result.get("answer") or ask_result.get("response", "")
conversation_id = ask_result.get("conversationId")
display(Markdown(ask_answer))

Ask a follow-up in the same conversation. Because you pass `conversationId`, Copilot knows that "the delivery decision" refers to the CALD-201 emails.

In [ ]:
follow_up = await call_workiq(
    "ask",
    {
        "question": "For the delivery decision, what are the two options and what does each mean for the 30 April delivery?",
        "conversationId": conversation_id,
    },
)
display(Markdown(follow_up.get("answer") or follow_up.get("response", "")))

## Step 4: Retrieve the evidence behind the answer

`ask` gives you a conclusion. `retrieve` gives you the **evidence**: grounding passages in Markdown, each with source metadata such as a link and a sensitivity label. An agent uses `retrieve` when it wants to reason over the passages itself and cite each claim.

The `capabilities` argument limits the search to specific sources. Here, it searches only email. The results are in `structuredContent["application/vnd.ms-workiq.retrieval"]`.

In [ ]:
retrieval = await call_workiq(
    "retrieve",
    {
        "query": "CALD-201 Summit Dose delivery decision, subcontractor approval, and equipment mapping",
        "capabilities": [{"name": "Email"}],
    },
)
payload = retrieval.get("application/vnd.ms-workiq.retrieval", retrieval)
results = payload.get("results", [])
print(f"{len(results)} results (stopped reason: {payload.get('stoppedReason')})\n")
for number, item in enumerate(results, 1):
    print(f"[{number}] " + item.get("markdown", "")[:600].strip())
    metadata = {key: value for key, value in item.items() if key != "markdown"}
    print("    metadata:", json.dumps(metadata, default=str)[:400], "\n")

## Step 5: Find resource paths and inspect a schema

For precise reads and writes, find the exact path first. `search_paths` returns matching Microsoft 365 resource paths with their supported operations, and `get_schema` describes the request or response for one path and operation.

The cell reads the argument name for `search_paths` from the tool's input schema that you listed in Step 2, so it keeps working if the server renames the argument.

In [ ]:
search_argument = next(iter(workiq_tools["search_paths"].inputSchema.get("required", ["query"])))
paths = await call_workiq("search_paths", {search_argument: "calendar events"})
for path in paths["paths"][:8]:
    print(path["path"], "->", ", ".join(path["operations"]))

schema = await call_workiq("get_schema", {"path": "/me/events", "operationType": "create", "format": "typescript"})
schema_text = schema if isinstance(schema, str) else json.dumps(schema, indent=2)
print("\nCreate schema for /me/events (first 1,500 characters):\n")
print(schema_text[:1500])

## Step 6: Fetch structured email data

`fetch` reads entities by relative path and returns data, not a synthesized answer. Request the ten most recent inbox messages with only the fields you need, then keep the CALD-201 messages. You will reuse their subjects when you schedule the decision meeting.

In [ ]:
fetched = await call_workiq(
    "fetch",
    {"entityUrls": ["/me/mailFolders/inbox/messages?$top=10&$select=id,subject,from,receivedDateTime"]},
)
messages = fetched["results"][0]["data"]["value"]
cald_messages = [message for message in messages if "CALD-201" in (message.get("subject") or "")]
for message in cald_messages:
    sender = message["from"]["emailAddress"]["name"]
    print(f"{message['receivedDateTime']} | {sender} | {message['subject']}")
if not cald_messages:
    print("No CALD-201 messages in the ten most recent inbox messages. Check Outlook, or ask a proctor.")

## Step 7: List the agents you can ask

`list_agents` returns the Microsoft 365 Copilot agents available to you. Pass an `agentId` from this list to `ask` or `retrieve` to route a question to that agent. The built-in Microsoft 365 Copilot agent is always included.

In [ ]:
agents = await call_workiq("list_agents")
print(json.dumps(agents, indent=2)[:1500])

---

## B: Act on your behalf

The next steps **change real Microsoft 365 data** in your lab mailbox and calendar without further prompts. Run them one at a time and check the result in Outlook after each. Each write targets only items that this notebook creates, and every email goes only to your own lab account. No fictional colleague receives an invitation or a message.

If a write fails with a policy error, your tenant blocks Work IQ write operations. Ask a proctor.

## Step 8: Schedule the delivery decision

`create_entity` posts JSON to a collection path. Create a 30-minute review tomorrow at 15:00 UTC with no attendees, using the CALD-201 subjects you fetched in Step 6 as the agenda. The cell captures the new event's ID so the next steps change only this event. Open the event in your Outlook calendar before continuing.

`jsonBody` is a JSON-encoded string, so the cell encodes the payload with `json.dumps`.

In [ ]:
from datetime import datetime, timedelta, timezone

if globals().get("event_id"):
    raise ValueError("The decision meeting was already created. Continue to the next step instead of creating a duplicate.")

review_subject = "CALD-201: decide on the weekend coating slot"
agenda = "\n".join(
    [
        "Decide whether to authorize Summit Dose's reserved weekend coating slot or accept the 12 May PPQ-01 release.",
        "The approved purchase order commits to a 30 April delivery. The emails are planning updates, not amendments.",
        "",
        "Source emails:",
        *[f"- {message['subject']}" for message in cald_messages],
    ]
)
start = (datetime.now(timezone.utc) + timedelta(days=1)).replace(hour=15, minute=0, second=0, microsecond=0)
end = start + timedelta(minutes=30)

created = await call_workiq(
    "create_entity",
    {
        "parentUrl": "/me/events",
        "jsonBody": json.dumps(
            {
                "subject": review_subject,
                "body": {"contentType": "Text", "content": agenda},
                "start": {"dateTime": start.strftime("%Y-%m-%dT%H:%M:%S"), "timeZone": "UTC"},
                "end": {"dateTime": end.strftime("%Y-%m-%dT%H:%M:%S"), "timeZone": "UTC"},
                "isReminderOn": True,
                "reminderMinutesBeforeStart": 15,
            }
        ),
    },
)
event = created.get("data", created)
event_id = event.get("id", "")
if not event_id:
    print(json.dumps(created, indent=2, default=str)[:1500])
    raise RuntimeError("The event was created, but its ID is missing from the response. Do not create it again.")
print(f"Created: {review_subject} | {start:%Y-%m-%d %H:%M} UTC | 30 minutes | no attendees")
print(f"Event ID: {event_id}")

## Step 9: Extend the meeting and check your reminders

`update_entity` changes only the fields you send. Add change-control planning to the subject and extend the meeting to 45 minutes. Refresh your calendar to see the change.

Then use `call_function`, which calls a read-only function instead of reading an entity. The Microsoft Graph `reminderView` function returns the reminders due in a time window, so you can confirm that the updated meeting will remind you.

In [ ]:
if not globals().get("event_id"):
    raise ValueError("Create the decision meeting in Step 8 before updating it.")

updated_subject = "CALD-201: decide on the weekend coating slot and change control"
await call_workiq(
    "update_entity",
    {
        "entityUrl": f"/me/events/{event_id}",
        "jsonBody": json.dumps(
            {
                "subject": updated_subject,
                "end": {"dateTime": (start + timedelta(minutes=45)).strftime("%Y-%m-%dT%H:%M:%S"), "timeZone": "UTC"},
            }
        ),
    },
)
print(f"Updated: {updated_subject} | 45 minutes")

window_start = (start - timedelta(hours=1)).strftime("%Y-%m-%dT%H:%M:%SZ")
window_end = (start + timedelta(hours=1)).strftime("%Y-%m-%dT%H:%M:%SZ")
reminders = await call_workiq(
    "call_function",
    {"functionUrl": f"/me/reminderView(startDateTime='{window_start}',endDateTime='{window_end}')"},
)
for reminder in reminders.get("data", {}).get("value", []):
    print(f"Reminder at {reminder['reminderFireTime']['dateTime']} UTC for '{reminder['eventSubject']}'")

## Step 10: Send a follow-up with the approved purchase order

`do_action` runs an action, such as sending, replying, or forwarding mail. Send yourself a follow-up that points out the difference between the planning update and the approved commitment, with **PO-CAL-SUM-2026-1108.pdf** attached from the lab's data folder. Check your inbox and Sent Items after running this cell. It can take a minute to arrive.

A successful send is not repeated in the same kernel.

In [ ]:
import base64
from pathlib import Path

if globals().get("follow_up_subject"):
    raise ValueError("The follow-up was already sent in this kernel. Check your inbox and Sent Items.")
if "@" not in lab_email:
    raise ValueError("The signed-in user's email address is missing. Rerun Step 1 with your lab account.")

purchase_order = Path("../../data/caldova/pdfs/PO-CAL-SUM-2026-1108.pdf")
purchase_order_bytes = purchase_order.read_bytes()
follow_up_subject = "CALD-201 follow-up: weekend coating slot decision"
follow_up_body = "\n".join(
    [
        "Lab follow-up based on the CALD-201 emails.",
        "",
        "The attached purchase order PO-CAL-SUM-2026-1108 is the approved commitment, with delivery on 30 April.",
        "Summit Dose's 12 May estimate is a planning update, not a purchase order amendment.",
        f"The decision meeting is scheduled for {start:%d %B %Y} at 15:00 UTC.",
    ]
)

await call_workiq(
    "do_action",
    {
        "actionUrl": "/me/sendMail",
        "jsonBody": json.dumps(
            {
                "message": {
                    "subject": follow_up_subject,
                    "body": {"contentType": "Text", "content": follow_up_body},
                    "toRecipients": [{"emailAddress": {"address": lab_email}}],
                    "attachments": [
                        {
                            "@odata.type": "#microsoft.graph.fileAttachment",
                            "name": purchase_order.name,
                            "contentType": "application/pdf",
                            "contentBytes": base64.b64encode(purchase_order_bytes).decode("ascii"),
                        }
                    ],
                },
                "saveToSentItems": True,
            }
        ),
    },
)
print(f"Send request accepted for {lab_email}: {follow_up_subject}")

## Step 11: Download the attachment with `fetch_blob`

`fetch` returns JSON. For binary content, such as attachments and files, use `fetch_blob`. It returns the bytes base64-encoded in `structuredContent`, together with the content type and size.

Find the sent message in Sent Items, list its attachments, download the purchase order, and confirm that the bytes match the file on disk. The cell retries for up to a minute while the message is saved.

In [ ]:
import asyncio

sent_message = None
for _ in range(12):
    sent = await call_workiq(
        "fetch",
        {"entityUrls": ["/me/mailFolders/sentitems/messages?$top=10&$select=id,subject,hasAttachments"]},
    )
    sent_message = next(
        (message for message in sent["results"][0]["data"]["value"] if message.get("subject") == follow_up_subject),
        None,
    )
    if sent_message:
        break
    await asyncio.sleep(5)
if sent_message is None:
    raise RuntimeError("The follow-up is not in Sent Items yet. Wait a moment and rerun this cell.")

attachments = await call_workiq(
    "fetch",
    {"entityUrls": [f"/me/messages/{sent_message['id']}/attachments?$select=id,name,size"]},
)
attachment = attachments["results"][0]["data"]["value"][0]
blob = await call_workiq("fetch_blob", {"path": f"/me/messages/{sent_message['id']}/attachments/{attachment['id']}/$value"})
downloaded = base64.b64decode(blob["base64Content"])

print(f"Downloaded {attachment['name']}: {blob.get('contentType')} | {len(downloaded):,} bytes")
print("Matches the file on disk:", downloaded == purchase_order_bytes)

## Step 12: Clean up the decision meeting

`delete_entity` removes one item by path. Delete only the meeting created in Step 8, using its captured ID. This doesn't touch the seeded emails, the follow-up email, or any other calendar events. Refresh your calendar to confirm that the meeting is gone.

Successful cleanup clears the ID, so rerunning Step 9 or this step can't reuse it.

In [ ]:
if not globals().get("event_id"):
    raise ValueError("There is no captured meeting to delete. Create one in Step 8 first.")

await call_workiq("delete_entity", {"entityUrl": f"/me/events/{event_id}"})
print(f"Deleted the notebook-created meeting: {event_id}")
event_id = ""

#### 🔍 Source Hunt

Compare what the different tools returned. Can you find:

1. Which owner and deadline in the `ask` answer from Step 3 can you trace to a passage in the `retrieve` results from Step 4? Open the original email in Outlook and compare.
2. Does any `retrieve` result carry a sensitivity label or a link in its metadata?
3. Which statement in your follow-up email comes from the purchase order rather than from the emails? Why does that difference matter?
4. In Outlook, find the follow-up in your inbox and open the attached purchase order. Is it the same file that `fetch_blob` downloaded?

## ✅ Mission complete

**What you built:**

* ✓ **Copilot tools**: `ask` synthesized an answer about the CALD-201 emails and kept the conversation for a follow-up, and `retrieve` returned the cited passages behind it, scoped to email.
* ✓ **Discovery and reads**: `search_paths` and `get_schema` found paths and schemas at runtime. `fetch`, `call_function`, and `fetch_blob` returned structured email data, your reminders, and a downloaded attachment.
* ✓ **Writes**: `create_entity`, `update_entity`, and `delete_entity` managed a decision meeting, and `do_action` sent a follow-up with the approved purchase order attached.

Every request ran as you. Only items created by this notebook were changed, and only your own lab account received mail. If you skipped cleanup, return to Step 12 to delete the meeting.

➡️ Return to lab instructions.